> WoC p2c returned 40,055 commit hashes. Commit metadata
> was retrieved for 40,013 hashes. The remaining 42
> returned “Key not found” from both commit.tch and the
> commit object endpoint.

In [1]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# The default synchronous client times out after 30 seconds, but WoC can
# ask clients to wait 60 seconds after a rate-limit response. Give its
# built-in retry handling enough time to finish.
class PatientWocMapsRemote(WocMapsRemote):
  def _asyncio_run(self, coro, timeout=300):
    return super()._asyncio_run(coro, timeout=timeout)

# creates the client
woc = PatientWocMapsRemote(base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = PatientWocMapsRemote(base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY")

# WoC project IDs assigned to tgarrio1 in net2prj.csv
projects = [
    'movsim_movsim',
    'iml-wg_hepml-livingreview',
    'juliagpu_cuda.jl',
    'soedinglab_hh-suite',
    'jdblischak_workflowr',
    'oscaribv_pyaneti',
    'isl-org_midas',
    'covid-projections_covid-data-model',
    'hive-researchgroup_volumetric-data-interaction',
    'antsx_antspy',
]
list_df_commits = []
for prj in projects:
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits, ignore_index=True)
df = df.drop_duplicates(subset=['sha1', 'project'])
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv', index=False)
df.head(1)

,sha1,project
0,0011be813bdf283cd3f816e3d8af12776a4206f6,movsim_movsim


In [2]:
# Fetch commit details in batches of 10. This cell can resume from both
# df_commit_data.csv and successful batches left in memory by a failed run.
import time
from pathlib import Path

BATCH_SIZE = 10
REQUEST_DELAY_SECONDS = 1.5
MAX_RETRIES = 3
RETRY_WAIT_SECONDS = 60
CHECKPOINT_EVERY = 25
checkpoint_path = Path('df_commit_data.csv')
raw_columns = [
    'commit', 'tree', 'parent', 'author', 'author_time', 'author_tz',
    'committer', 'committer_time', 'committer_tz', 'message',
]

# Preserve progress from the failed run if this kernel has not restarted.
saved_frames = []
in_memory_rows = globals().get('commit_data', [])
if in_memory_rows:
  saved_frames.append(pd.DataFrame(in_memory_rows))

# Also reuse the commit details saved by an earlier run.
if checkpoint_path.exists():
  saved_data = pd.read_csv(checkpoint_path)
  if set(raw_columns).issubset(saved_data.columns):
    saved_frames.append(saved_data[raw_columns])

if saved_frames:
  raw_commit_data = (
      pd.concat(saved_frames, ignore_index=True)
      .dropna(subset=['commit'])
      .drop_duplicates(subset=['commit'])
  )
else:
  raw_commit_data = pd.DataFrame(columns=raw_columns)

commit_data = raw_commit_data[raw_columns].to_dict('records')
completed_commits = set(raw_commit_data['commit'].astype(str))
unique_sha1s = df['sha1'].drop_duplicates().astype(str).tolist()
pending_sha1s = [sha1 for sha1 in unique_sha1s if sha1 not in completed_commits]
chunks = [
    pending_sha1s[x:x + BATCH_SIZE]
    for x in range(0, len(pending_sha1s), BATCH_SIZE)
]
errors = {}
print(f'Resuming with {len(completed_commits):,} commits already saved.')
print(f'{len(pending_sha1s):,} unique commits remain in {len(chunks):,} batches.')

for chunk_number, chunk in enumerate(tqdm(chunks), start=1):
  for attempt in range(1, MAX_RETRIES + 1):
    try:
      # commit.tch returns the same data as commit, but faster.
      res, err = woc.get_values_many('commit.tch', chunk)
      break
    except Exception as exc:
      if attempt == MAX_RETRIES:
        pd.DataFrame(commit_data, columns=raw_columns).to_csv(
            checkpoint_path, index=False
        )
        raise
      wait_seconds = RETRY_WAIT_SECONDS * attempt
      print(
          f'Batch timed out ({type(exc).__name__}); '
          f'waiting {wait_seconds}s before retry {attempt + 1}/{MAX_RETRIES}.'
      )
      time.sleep(wait_seconds)

  res = {key: value[0] for key, value in res.items()}

  # Some hashes appear in p2c but are absent from the commit.tch map.
  # Retry those hashes through WoC's commit object endpoint. Unlike
  # commit.tch, show_content_many already returns the commit tuple directly.
  if err:
    fallback_keys = list(err)
    print(
        f'{len(fallback_keys)} hashes missing from commit.tch; '
        'trying the commit object endpoint.'
    )
    for fallback_attempt in range(1, MAX_RETRIES + 1):
      try:
        fallback_res, fallback_err = woc.show_content_many(
            'commit', fallback_keys
        )
        res.update(fallback_res)
        err = fallback_err
        break
      except Exception as exc:
        if fallback_attempt == MAX_RETRIES:
          print(
              'Commit object fallback failed after '
              f'{MAX_RETRIES} attempts: {type(exc).__name__}: {exc}'
          )
          break
        wait_seconds = RETRY_WAIT_SECONDS * fallback_attempt
        print(
            f'Fallback timed out ({type(exc).__name__}); waiting '
            f'{wait_seconds}s before retry '
            f'{fallback_attempt + 1}/{MAX_RETRIES}.'
        )
        time.sleep(wait_seconds)

  if err:
    print('Unresolved WoC errors', err)
    errors.update(err)

  for commit_sha, commit in res.items():
    commit_data.append({
        'commit': commit_sha,
        'tree': commit[0],
        'parent': list(commit[1]),
        'author': commit[2][0],
        'author_time': int(commit[2][1]),
        'author_tz': commit[2][2],
        'committer': commit[3][0],
        'committer_time': int(commit[3][1]),
        'committer_tz': commit[3][2],
        'message': commit[4],
    })

  if chunk_number % CHECKPOINT_EVERY == 0:
    pd.DataFrame(commit_data, columns=raw_columns).to_csv(
        checkpoint_path, index=False
    )
  time.sleep(REQUEST_DELAY_SECONDS)

# Save the completed raw cache, then restore project membership by merging.
raw_commit_data = (
    pd.DataFrame(commit_data, columns=raw_columns)
    .drop_duplicates(subset=['commit'])
)
raw_commit_data.to_csv(checkpoint_path, index=False)
unresolved_sha1s = sorted(set(unique_sha1s) - set(raw_commit_data['commit']))
print(f'Retrieved {len(raw_commit_data):,} of {len(unique_sha1s):,} unique commits.')
print(f'Unresolved after both WoC endpoints: {len(unresolved_sha1s):,}')
df_commit_data = raw_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.head(2)

Resuming with 40,013 commits already saved.
42 unique commits remain in 5 batches.


  0%|          | 0/5 [00:00<?, ?it/s]

10 hashes missing from commit.tch; trying the commit object endpoint.
Unresolved WoC errors {'6ea1529d8dc1a79e2529639ceae2161c0846005f': 'Key 6ea1529d8dc1a79e2529639ceae2161c0846005f not found in /da5_fast/All.sha1c/commit_110.tch', '9bb5d5c05c91c0bc8c8572d4ab1573bb8b71d6c1': 'Key 9bb5d5c05c91c0bc8c8572d4ab1573bb8b71d6c1 not found in /da5_fast/All.sha1c/commit_27.tch', 'c2cc260d11842c38746d8a262854c06d1415eacf': 'Key c2cc260d11842c38746d8a262854c06d1415eacf not found in /da5_fast/All.sha1c/commit_66.tch', 'cf97c0acb40451bdf6db7a51c533a5c33362057c': 'Key cf97c0acb40451bdf6db7a51c533a5c33362057c not found in /da5_fast/All.sha1c/commit_79.tch', 'e1c71148d617cce68705197b1c6ecafe91fdf84c': 'Key e1c71148d617cce68705197b1c6ecafe91fdf84c not found in /da5_fast/All.sha1c/commit_97.tch', '00f9998b87b1f6265d01493355d14996e47f7747': 'Key 00f9998b87b1f6265d01493355d14996e47f7747 not found in /da5_fast/All.sha1c/commit_0.tch', '0b62b92db29e59aefe1ba6be665f33e77ab26e2e': 'Key 0b62b92db29e59aefe1ba6be

 20%|██        | 1/5 [00:01<00:06,  1.59s/it]

10 hashes missing from commit.tch; trying the commit object endpoint.
Unresolved WoC errors {'3f888d7945dc624e8aee7bef313795ff07fbf54e': 'Key 3f888d7945dc624e8aee7bef313795ff07fbf54e not found in /da5_fast/All.sha1c/commit_63.tch', '46e304a874062b612426284950b045d5fbe3df15': 'Key 46e304a874062b612426284950b045d5fbe3df15 not found in /da5_fast/All.sha1c/commit_70.tch', '4d7a289f28e3a4929c55eab6e8edd4c682be298f': 'Key 4d7a289f28e3a4929c55eab6e8edd4c682be298f not found in /da5_fast/All.sha1c/commit_77.tch', '74edf453bd124850ec7e2f3aedf46cac2a61678c': 'Key 74edf453bd124850ec7e2f3aedf46cac2a61678c not found in /da5_fast/All.sha1c/commit_116.tch', 'afef88bf309fb8c51762b1c3ef2f0deaef0b2bbd': 'Key afef88bf309fb8c51762b1c3ef2f0deaef0b2bbd not found in /da5_fast/All.sha1c/commit_47.tch', '0d6046f490ea200d65943ba24a1ac7581bf242dc': 'Key 0d6046f490ea200d65943ba24a1ac7581bf242dc not found in /da5_fast/All.sha1c/commit_13.tch', '19b08e85b28156ecb1e4fda2896f5f0b79e4e28b': 'Key 19b08e85b28156ecb1e4fda

...

 80%|████████  | 4/5 [00:06<00:01,  1.59s/it]

2 hashes missing from commit.tch; trying the commit object endpoint.
Unresolved WoC errors {'0dd5a6cf5576af49475204bcd4b4a0d524cc21a9': 'Key 0dd5a6cf5576af49475204bcd4b4a0d524cc21a9 not found in /da5_fast/All.sha1c/commit_13.tch', '5bf48ebb766e61e1df0c05490ca394cacc55ae01': 'Key 5bf48ebb766e61e1df0c05490ca394cacc55ae01 not found in /da5_fast/All.sha1c/commit_91.tch'}


100%|██████████| 5/5 [00:07<00:00,  1.59s/it]

Retrieved 40,013 of 40,055 unique commits.
Unresolved after both WoC endpoints: 42


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0011be813bdf283cd3f816e3d8af12776a4206f6,b50265945528e4917a893a37ff5d91c170c6cfbd,['2d6658b2ef06464904095a7e5f377a6b04ae8126'],rgerm <germ@ralphgerm.de>,1330085492,100,rgerm <germ@ralphgerm.de>,1330085492,100,Added possibility of different traffic composi...,0011be813bdf283cd3f816e3d8af12776a4206f6,movsim_movsim
1,00331773b48e4e1d82c03fb907b6b59a923461eb,4e191ed34ea78e3f3d914e5456e5a4267c1dc943,['4e7442388db9da480fa60d693fa3c771da79639f'],akegermany <mail@akesting.de>,1424971946,100,akegermany <mail@akesting.de>,1424971946,100,minor code cleaning\n,00331773b48e4e1d82c03fb907b6b59a923461eb,movsim_movsim


In [3]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0011be813bdf283cd3f816e3d8af12776a4206f6', 'tree': 'b50265945528e4917a893a37ff5d91c170c6cfbd', 'parent': "['2d6658b2ef06464904095a7e5f377a6b04ae8126']", 'author': 'rgerm <germ@ralphgerm.de>', 'author_time': 1330085492, 'author_tz': 100, 'committer': 'rgerm <germ@ralphgerm.de>', 'committer_time': 1330085492, 'committer_tz': 100, 'message': 'Added possibility of different traffic compositions for\neach road with icMacro.\n'}


In [4]:
# Select and rename the five columns required for the Part 1 submission.
# Project membership comes from the merge above, so every row keeps the
# correct project instead of using the final value of the loop variable.
dfinf = (
    df_commit_data[['project', 'commit', 'author', 'author_time', 'message']]
    .rename(columns={
        'project': 'project_wocid',
        'commit': 'commit_sha1',
        'author_time': 'time',
        'message': 'commit message',
    })
    .drop_duplicates(subset=['project_wocid', 'commit_sha1'])
)


In [5]:
# check if it has the right content
dfinf.head(1)

,project_wocid,commit_sha1,author,time,commit message
0,movsim_movsim,0011be813bdf283cd3f816e3d8af12776a4206f6,rgerm <germ@ralphgerm.de>,1330085492,Added possibility of different traffic composi...


In [6]:
# Write the complete file once so rerunning this cell cannot append duplicates.
yournetid = 'tgarrio1'
dfinf.to_csv(yournetid + '_project_summary.csv', index=False, sep=';')
print(f'Wrote {len(dfinf):,} rows for {dfinf["project_wocid"].nunique()} projects.')

Wrote 40,013 rows for 10 projects.


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Part 1 Project Summary

GitHub values retrieved on 2026-09-24.

| Project (WoC ID) | WoC commits | WoC authors | Min time (Unix / UTC) | Max time (Unix / UTC) | GitHub stars | GitHub forks | Latest GitHub commit date |
|---|---:|---:|---|---|---:|---:|---|
| [movsim_movsim](https://github.com/movsim/movsim) | 2539 | 23 | 1302020322 / 2011-04-05 16:18:42 UTC | 1762103089 / 2025-11-02 17:04:49 UTC | 340 | 93 | 2026-09-02 |
| [iml-wg_hepml-livingreview](https://github.com/iml-wg/HEPML-LivingReview) | 877 | 96 | 1587584663 / 2020-04-22 19:44:23 UTC | 1762203246 / 2025-11-03 20:54:06 UTC | 453 | 136 | 2026-08-10 |
| [juliagpu_cuda.jl](https://github.com/JuliaGPU/CUDA.jl) | 14323 | 381 | 1378553736 / 2013-09-07 11:35:36 UTC | 1762624702 / 2025-11-08 17:58:22 UTC | 1428 | 282 | 2026-09-24 |
| [soedinglab_hh-suite](https://github.com/soedinglab/hh-suite) | 1495 | 98 | 1226577471 / 2008-11-13 11:57:51 UTC | 1755607024 / 2025-08-19 12:37:04 UTC | 628 | 152 | 2025-08-12 |
| [jdblischak_workflowr](https://github.com/jdblischak/workflowr) | 1424 | 31 | 1481147440 / 2016-12-07 21:50:40 UTC | 1754099317 / 2025-08-02 01:48:37 UTC | 907 | 107 | 2026-07-01 |
| [oscaribv_pyaneti](https://github.com/oscaribv/pyaneti) | 662 | 5 | 1453285532 / 2016-01-20 10:25:32 UTC | 1762449400 / 2025-11-06 17:16:40 UTC | 53 | 16 | 2026-06-03 |
| [isl-org_midas](https://github.com/isl-org/MiDaS) | 200 | 41 | 1561385316 / 2019-06-24 14:08:36 UTC | 1755784085 / 2025-08-21 13:48:05 UTC | 5418 | 726 | 2024-08-23 |
| [covid-projections_covid-data-model](https://github.com/covid-projections/covid-data-model) | 14427 | 69 | 1584466828 / 2020-03-17 17:40:28 UTC | 1712160037 / 2024-04-03 16:00:37 UTC | 149 | 53 | 2026-02-15 |
| [hive-researchgroup_volumetric-data-interaction](https://github.com/HIVE-ResearchGroup/volumetric-data-interaction) | 1589 | 10 | 1630482898 / 2021-09-01 07:54:58 UTC | 1727767776 / 2024-10-01 07:29:36 UTC | 3 | 1 | 2024-10-01 |
| [antsx_antspy](https://github.com/ANTsX/ANTsPy) | 2519 | 65 | 1503943258 / 2017-08-28 18:00:58 UTC | 1773735351 / 2026-03-17 08:15:51 UTC | 888 | 179 | 2026-08-19 |

In [7]:
# Validate the result and calculate the WoC values needed in the notebook summary.
assert dfinf['project_wocid'].nunique() == len(projects)
assert not dfinf.duplicated(['project_wocid', 'commit_sha1']).any()
assert dfinf[['commit_sha1', 'author', 'time']].notna().all().all()

project_stats = (
    dfinf.groupby('project_wocid')
    .agg(
        number_of_commits=('commit_sha1', 'nunique'),
        number_of_authors=('author', 'nunique'),
        min_time=('time', 'min'),
        max_time=('time', 'max'),
    )
    .reset_index()
)
project_stats['min_date_utc'] = pd.to_datetime(project_stats['min_time'], unit='s', utc=True)
project_stats['max_date_utc'] = pd.to_datetime(project_stats['max_time'], unit='s', utc=True)
project_stats

,project_wocid,number_of_commits,number_of_authors,min_time,max_time,min_date_utc,max_date_utc
0,antsx_antspy,2517,65,1503943258,1773735351,2017-08-28 18:00:58+00:00,2026-03-17 08:15:51+00:00
1,covid-projections_covid-data-model,14427,69,1584466828,1712160037,2020-03-17 17:40:28+00:00,2024-04-03 16:00:37+00:00
2,hive-researchgroup_volumetric-data-interaction,1589,10,1630482898,1727767776,2021-09-01 07:54:58+00:00,2024-10-01 07:29:36+00:00
3,iml-wg_hepml-livingreview,877,96,1587584663,1762203246,2020-04-22 19:44:23+00:00,2025-11-03 20:54:06+00:00
4,isl-org_midas,200,41,1561385316,1755784085,2019-06-24 14:08:36+00:00,2025-08-21 13:48:05+00:00
5,jdblischak_workflowr,1399,31,1481147440,1754099317,2016-12-07 21:50:40+00:00,2025-08-02 01:48:37+00:00
6,juliagpu_cuda.jl,14323,381,1378553736,1762624702,2013-09-07 11:35:36+00:00,2025-11-08 17:58:22+00:00
7,movsim_movsim,2534,23,1302020322,1762103089,2011-04-05 16:18:42+00:00,2025-11-02 17:04:49+00:00
8,oscaribv_pyaneti,662,5,1453285532,1762449400,2016-01-20 10:25:32+00:00,2025-11-06 17:16:40+00:00
9,soedinglab_hh-suite,1485,98,1226577471,1755607024,2008-11-13 11:57:51+00:00,2025-08-19 12:37:04+00:00
